In [ ]:
import numpy as np
import pandas as pd
import re
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error

warnings.filterwarnings('ignore')
np.random.seed(42)
print('Libraries loaded.')

## 1. Load Data

In [ ]:
DATA_PATH   = '/kaggle/input/competitions/finclub-open-project-26/dataset.csv'
SAMPLE_PATH = '/kaggle/input/competitions/finclub-open-project-26/sandbox_solution.csv'
OUTPUT_PATH = '/kaggle/working/submission.csv'
FILLED_PATH = '/kaggle/working/filled_dataset.csv'
SEPARATOR   = '||'

df = pd.read_csv(DATA_PATH)
df['datetime'] = pd.to_datetime(df['datetime'], dayfirst=True)
df = df.sort_values('datetime').reset_index(drop=True)

feature_cols = [c for c in df.columns if c not in ['datetime', 'underlying_price']]

print(f'Shape          : {df.shape}')
print(f'Date range     : {df["datetime"].min()} → {df["datetime"].max()}')
print(f'Option columns : {len(feature_cols)}  '
      f'({sum(c.endswith("CE") for c in feature_cols)} CE + '
      f'{sum(c.endswith("PE") for c in feature_cols)} PE)')
print(f'Spot range     : {df["underlying_price"].min():.0f} – '
      f'{df["underlying_price"].max():.0f}')
print(f'Missing cells  : {df[feature_cols].isnull().sum().sum():,}  '
      f'({df[feature_cols].isnull().mean().mean():.1%})')
df.head(3)

## 2. Parse Column Metadata




In [ ]:
MONTH_MAP = {
    'JAN': 1, 'FEB': 2, 'MAR': 3, 'APR': 4, 'MAY': 5,  'JUN': 6,
    'JUL': 7, 'AUG': 8, 'SEP': 9, 'OCT': 10,'NOV': 11, 'DEC': 12,
}

def parse_col(col):
    """Extract (strike, opt_type, expiry) — full regex avoids the year-digit ambiguity."""
    m = re.search(r'(\d{2})([A-Z]{3})(\d{2})(\d+)(CE|PE)$', col)
    if not m:
        raise ValueError(f'Cannot parse: {col!r}')
    expiry   = pd.Timestamp(year=2000 + int(m.group(3)),
                             month=MONTH_MAP[m.group(2)],
                             day=int(m.group(1)))
    strike   = int(m.group(4))
    opt_type = 1 if m.group(5) == 'CE' else 0
    return strike, opt_type, expiry

col_info = {col: parse_col(col) for col in feature_cols}

# Verify moneyness is near 1.0
spot0 = df['underlying_price'].iloc[0]
print('Sanity check — moneyness must be near 1.0:')
for col in feature_cols[:6]:
    s, o, e = col_info[col]
    print(f'  {col:<32} → strike={s:>6}  moneyness={s/spot0:.4f}')

strikes  = sorted(set(v[0] for v in col_info.values()))
expiries = sorted(set(v[2] for v in col_info.values()))
print(f'\nStrikes  : {strikes[0]} – {strikes[-1]}  ({len(strikes)} total)')
print(f'Expiries : {[str(e.date()) for e in expiries]}')

## 3. EDA

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

sns.heatmap(df[feature_cols].isnull(), cbar=False,
            yticklabels=False, xticklabels=False, ax=axes[0])
axes[0].set_title('Missingness (rows=time, cols=strikes)')
axes[0].set_xlabel('Strike Columns'); axes[0].set_ylabel('Timestamps')

iv_obs = df[feature_cols].values.flatten()
iv_obs = iv_obs[~np.isnan(iv_obs)]
axes[1].hist(iv_obs, bins=60, color='steelblue', edgecolor='white', lw=0.3)
axes[1].axvline(np.median(iv_obs), color='red', linestyle='--',
                label=f'Median = {np.median(iv_obs):.3f}')
axes[1].set_title('Observed IV Distribution')
axes[1].set_xlabel('Implied Volatility'); axes[1].legend()

# ATM IV over time — col_info only
atm_rows = []
for idx, row in df.iterrows():
    spot = row['underlying_price']
    best_col, best_dist = None, np.inf
    for col in feature_cols:
        strike, opt_type, _ = col_info[col]
        if opt_type != 1: continue
        dist = abs(strike - spot)
        if dist < best_dist and not np.isnan(row[col]):
            best_dist = dist; best_col = col
    if best_col:
        atm_rows.append({'datetime': row['datetime'], 'atm_iv': row[best_col]})

atm_df = pd.DataFrame(atm_rows)
axes[2].plot(atm_df['datetime'], atm_df['atm_iv'], lw=0.8, color='darkorange')
axes[2].set_title('ATM Call IV Over Time')
axes[2].set_xlabel('Date'); axes[2].set_ylabel('IV')
axes[2].tick_params(axis='x', rotation=30)
plt.tight_layout(); plt.show()

print(f'Observed IV — mean: {iv_obs.mean():.4f} | std: {iv_obs.std():.4f} | '
      f'min: {iv_obs.min():.4f} | max: {iv_obs.max():.4f}')

In [ ]:
def plot_smile(df, col_info, feature_cols, timestamp_idx=300):
    row  = df.iloc[timestamp_idx]
    spot = row['underlying_price']
    data = {'CE': [], 'PE': []}
    for col in feature_cols:
        strike, opt_type, _ = col_info[col]
        opt_str = 'CE' if opt_type == 1 else 'PE'
        if not np.isnan(row[col]):
            data[opt_str].append((strike / spot, row[col]))
    fig, ax = plt.subplots(figsize=(10, 4))
    for ot, color in [('CE', 'steelblue'), ('PE', 'tomato')]:
        if data[ot]:
            x, y = zip(*sorted(data[ot]))
            ax.plot(x, y, 'o-', color=color, label=ot, lw=1.5, ms=4)
    ax.axvline(1.0, color='gray', linestyle='--', alpha=0.6, label='ATM (K/S = 1)')
    ax.set_xlabel('Moneyness (K / S)'); ax.set_ylabel('Implied Volatility')
    ax.set_title(f'Volatility Smile — '
                 f'{row["datetime"].strftime("%Y-%m-%d %H:%M")} | Spot: {spot:.0f}')
    ax.legend(); plt.tight_layout(); plt.show()

plot_smile(df, col_info, feature_cols, timestamp_idx=300)

## 4. Validation

Before predicting, we validate the method with two CV modes to understand
expected Kaggle performance.

### Mode 1 — Hold-out CV
Mask 20% of known cells, predict, measure MSE.
Optimistic: all other strikes at that timestamp are available.

### Mode 2 — Realistic CV
Mask the same *count* of strikes as are actually missing at each timestamp.



In [ ]:
def cs_poly_pred(xs, ys, strike_target, degrees=[3, 2, 1]):
    """
    Fit a polynomial to (strike, IV) pairs and evaluate at strike_target.
    Falls back to lower degree if not enough points. Clips to [0.01, 5.0].
    """
    xs = np.array(xs, dtype=float)
    ys = np.array(ys, dtype=float)
    for deg in degrees:
        if len(xs) >= deg + 1:
            try:
                coef = np.polyfit(xs, ys, deg)
                return float(np.clip(np.polyval(coef, strike_target), 0.01, 5.0))
            except np.linalg.LinAlgError:
                continue
    return float(np.mean(ys)) if len(ys) > 0 else 0.15

# Time interpolation (both directions) — used as blend component
filled_ti = df.copy()
for col in feature_cols:
    filled_ti[col] = (
        filled_ti[col]
        .interpolate(method='linear', limit_direction='both')
        .clip(lower=0.01, upper=5.0)
    )

W_CS = 0.15   # cross-sectional weight (optimised by realistic CV)
W_TI = 1 - W_CS

known_mask   = df[feature_cols].notna()
miss_per_row = df[feature_cols].isnull().sum(axis=1)

#  Mode 1: hold-out CV 
np.random.seed(42)
mse_holdout = []
for col in feature_cols:
    known_idx = df.index[known_mask[col]].tolist()
    if len(known_idx) < 20: continue
    test_idx  = np.random.choice(known_idx, size=max(1, len(known_idx)//5), replace=False)
    true_vals = df.loc[test_idx, col].values
    strike_t, opt_type, _ = col_info[col]
    preds = []
    for idx in test_idx:
        xs, ys = [], []
        for c2 in feature_cols:
            if c2 == col: continue
            s2, o2, _ = col_info[c2]
            if o2 != opt_type: continue
            v2 = df.loc[idx, c2]
            if not np.isnan(v2): xs.append(s2); ys.append(v2)
        cs_p = cs_poly_pred(xs, ys, strike_t)
        ti_p = float(filled_ti.loc[idx, col])
        preds.append(W_CS * cs_p + W_TI * ti_p)
    mse_holdout.append(mean_squared_error(true_vals, preds))

print(f'Hold-out CV MSE    : {np.mean(mse_holdout):.6f} ± {np.std(mse_holdout):.6f}')

#  Mode 2: realistic CV 
np.random.seed(42)
mse_realistic = []
for ts_idx in df.index:
    n_miss = miss_per_row[ts_idx]
    if n_miss == 0: continue
    known_cols = [c for c in feature_cols if not np.isnan(df.loc[ts_idx, c])]
    n_test = min(n_miss, max(1, len(known_cols) // 4))
    if len(known_cols) < n_test + 3: continue
    test_cols = np.random.choice(known_cols, size=n_test, replace=False)
    true_vals = np.array([df.loc[ts_idx, c] for c in test_cols])
    preds = []
    for col in test_cols:
        strike_t, opt_type, _ = col_info[col]
        xs, ys = [], []
        for c2 in feature_cols:
            if c2 == col or c2 in test_cols: continue
            s2, o2, _ = col_info[c2]
            if o2 != opt_type: continue
            v2 = df.loc[ts_idx, c2]
            if not np.isnan(v2): xs.append(s2); ys.append(v2)
        cs_p = cs_poly_pred(xs, ys, strike_t)
        ti_p = float(filled_ti.loc[ts_idx, col])
        preds.append(W_CS * cs_p + W_TI * ti_p)
    mse_realistic.append(mean_squared_error(true_vals, preds))

print(f'Realistic CV MSE   : {np.mean(mse_realistic):.6f} ± {np.std(mse_realistic):.6f}')
print(f'\nNote: Realistic CV is closer to Kaggle public score (~0.000627).')
print(f'Hold-out CV is optimistic because it has all other strikes available.')

## 5. Fill All Missing Values




In [ ]:
filled_final = filled_ti.copy()   # start from time-interp fill

n_cs_used = 0
n_ti_only = 0

for col in feature_cols:
    miss_idx = df.index[df[col].isna()]
    if len(miss_idx) == 0:
        continue
    strike_t, opt_type, _ = col_info[col]

    for idx in miss_idx:
        # All observed strikes of same type at this timestamp
        xs, ys = [], []
        for c2 in feature_cols:
            if c2 == col: continue
            s2, o2, _ = col_info[c2]
            if o2 != opt_type: continue
            v2 = df.loc[idx, c2]   # original observed only
            if not np.isnan(v2):
                xs.append(s2)
                ys.append(v2)

        ti_p = float(filled_ti.loc[idx, col])

        if xs:
            cs_p = cs_poly_pred(xs, ys, strike_t)
            filled_final.loc[idx, col] = W_CS * cs_p + W_TI * ti_p
            n_cs_used += 1
        else:
            # Fallback: pure time-interp (never happens with this dataset)
            filled_final.loc[idx, col] = ti_p
            n_ti_only += 1

print(f'Cells filled (CS blend) : {n_cs_used:,}')
print(f'Cells filled (TI only)  : {n_ti_only:,}')
print(f'NaN remaining           : {filled_final[feature_cols].isnull().sum().sum()}')

## 6. Sanity Checks

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

pred_vals = [float(filled_final.loc[idx, col])
             for col in feature_cols
             for idx in df.index[df[col].isna()]]
obs_vals  = iv_obs  # computed earlier

axes[0].hist(obs_vals,  bins=50, alpha=0.6, color='steelblue',
             label='Observed', density=True)
axes[0].hist(pred_vals, bins=50, alpha=0.6, color='tomato',
             label='Predicted', density=True)
axes[0].set_title('IV Distribution: Observed vs Predicted')
axes[0].set_xlabel('Implied Volatility'); axes[0].legend()

print(f'Predicted IV — mean: {np.mean(pred_vals):.4f} | '
      f'min: {np.min(pred_vals):.4f} | max: {np.max(pred_vals):.4f}')
print(f'Observed  IV — mean: {np.mean(obs_vals):.4f} | '
      f'min: {np.min(obs_vals):.4f} | max: {np.max(obs_vals):.4f}')

# Smile reconstruction at most-missing timestamp
miss_per_ts = df[feature_cols].isnull().sum(axis=1)
ts_idx      = miss_per_ts.idxmax()
spot        = df.loc[ts_idx, 'underlying_price']

data_orig, data_fill = {}, {}
for col in feature_cols:
    strike, opt_type, _ = col_info[col]
    if opt_type != 1: continue      # CE only
    k = strike / spot
    if not np.isnan(df.loc[ts_idx, col]):
        data_orig[k] = df.loc[ts_idx, col]
    data_fill[k] = float(filled_final.loc[ts_idx, col])

xf, yf = zip(*sorted(data_fill.items()))
xo, yo = zip(*sorted(data_orig.items())) if data_orig else ([], [])

axes[1].plot(xf, yf, 'o-', color='steelblue', lw=1.5, ms=4, label='After fill')
if xo:
    axes[1].plot(xo, yo, 'rs', ms=6, label='Original (observed)', zorder=5)
axes[1].axvline(1.0, color='gray', linestyle='--', alpha=0.6)
axes[1].set_title(f'Smile Reconstruction — timestamp {ts_idx}\n'
                  f'({miss_per_ts[ts_idx]} missing | Spot = {spot:.0f})')
axes[1].set_xlabel('Moneyness (K / S)'); axes[1].set_ylabel('Implied Volatility')
axes[1].legend()
plt.tight_layout(); plt.show()

## 7. Submission

In [ ]:
import os

pred_lookup = {}
for col in feature_cols:
    for idx in df.index[df[col].isna()]:
        dt_str = df.loc[idx, 'datetime'].strftime('%d-%m-%Y %H:%M')
        uid    = f'{dt_str}{SEPARATOR}{col}'
        pred_lookup[uid] = float(filled_final.loc[idx, col])

print(f'Predictions built: {len(pred_lookup):,}')

if os.path.exists(SAMPLE_PATH):
    sample_sub          = pd.read_csv(SAMPLE_PATH)
    submission          = sample_sub[['id']].copy()
    submission['value'] = submission['id'].map(pred_lookup)
    n_unmatched         = submission['value'].isna().sum()
    print(f'Rows: {len(submission):,}  |  Unmatched IDs: {n_unmatched}  (must be 0)')
    if n_unmatched > 0:
        print('WARNING: ID mismatch — check strftime format')
        submission['value'] = submission['value'].fillna(submission['value'].median())
else:
    print('sandbox_solution.csv not found — building IDs independently')
    rows_out   = [{'id': k, 'value': v} for k, v in pred_lookup.items()]
    submission = pd.DataFrame(rows_out).sort_values('id').reset_index(drop=True)

submission.to_csv(OUTPUT_PATH, index=False)
print(f'\nsubmission.csv saved  : {OUTPUT_PATH}')
print(f'Rows        : {len(submission):,}')
print(f'Value range : {submission["value"].min():.5f} – {submission["value"].max():.5f}')
print(f'Null values : {submission["value"].isna().sum()}')
submission.head(8)

In [ ]:
# Also save wide-format (compatible with submission-converter.ipynb)
filled_wide = filled_final.copy()
filled_wide['datetime'] = df['datetime'].dt.strftime('%d-%m-%Y %H:%M')
filled_wide.to_csv(FILLED_PATH, index=False)
print(f'filled_dataset.csv saved: {FILLED_PATH}')
print(f'NaN remaining: {filled_wide[feature_cols].isnull().sum().sum()}')